# 11 - RAG Avanzado: Agentic RAG

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2.5-3 horas

---

## Índice

1. [¿Qué es Agentic RAG?](#intro)
2. [Arquitectura completa](#arquitectura)
3. [Implementación paso a paso](#implementacion)
4. [Evaluación con RAGAS](#evaluacion)
5. [Mejores prácticas de producción](#produccion)
6. [Ejercicio final](#ejercicio)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Diseñar sistemas RAG agentic completos
- Implementar evaluación de calidad con RAGAS
- Aplicar patrones de producción
- Optimizar el rendimiento del sistema

<a name="intro"></a>
## 1. ¿Qué es Agentic RAG?

**Agentic RAG** combina las capacidades de:
- **RAG**: Acceso a conocimiento externo
- **Agentes**: Razonamiento y uso de herramientas
- **Grafos**: Flujos de trabajo complejos

### Comparación

| RAG Simple | Agentic RAG |
|------------|-------------|
| Un paso de retrieval | Múltiples pasos adaptativos |
| Sin razonamiento | Planifica y razona |
| Query fija | Reformula queries |
| Sin verificación | Auto-corrige respuestas |

In [7]:
#!pip install -q langchain langchain-groq langgraph langchain-huggingface faiss-cpu

In [8]:
import os
from getpass import getpass
import warnings
warnings.filterwarnings('ignore')

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("GROQ API Key: ")

print("Configurado ✓")

Configurado ✓


<a name="arquitectura"></a>
## 2. Arquitectura Completa

```
┌────────────────────────────────────────────────────────────────┐
│                      AGENTIC RAG SYSTEM                        │
├────────────────────────────────────────────────────────────────┤
│                                                                │
│  Query → [Clasificar] → [Reformular?] → [Retrieval]           │
│              ↓                              ↓                  │
│     ┌───────┴───────┐              [Verificar contexto]       │
│     │   FAQ?        │                      ↓                  │
│     │   General?    │              [Generar respuesta]        │
│     │   Técnica?    │                      ↓                  │
│     └───────────────┘              [Verificar calidad]        │
│                                            ↓                  │
│                                    [Corregir si necesario]    │
│                                            ↓                  │
│                                    [Respuesta final]          │
└────────────────────────────────────────────────────────────────┘
```

<a name="implementacion"></a>
## 3. Implementación paso a paso

In [9]:
# Setup components
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List, Literal

# Initialize LLM
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.3)

# Create knowledge base
knowledge_docs = [
    Document(page_content="El IBI se paga anualmente y grava bienes inmuebles urbanos y rústicos.", 
             metadata={"source": "ibi", "type": "definition"}),
    Document(page_content="El IVTM grava la titularidad de vehículos. Se paga mientras esté matriculado.",
             metadata={"source": "ivtm", "type": "definition"}),
    Document(page_content="Bonificaciones IBI: familias numerosas hasta 90%, energías renovables 50%.",
             metadata={"source": "ibi", "type": "bonus"}),
    Document(page_content="Bonificaciones IVTM: vehículos eléctricos 75%, históricos 100%.",
             metadata={"source": "ivtm", "type": "bonus"}),
    Document(page_content="Plazos de pago: IBI segundo semestre, IVTM primer trimestre.",
             metadata={"source": "general", "type": "deadlines"}),
    Document(page_content="Recursos: Puede interponer recurso en 30 días desde la notificación.",
             metadata={"source": "general", "type": "appeals"}),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(knowledge_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Componentes inicializados ✓")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Componentes inicializados ✓


In [10]:
# Define state
class AgenticRAGState(TypedDict):
    question: str
    question_type: str
    reformulated_question: str
    context: str
    context_sufficient: bool
    response: str
    response_quality: str
    final_response: str
    iteration: int

# Node functions
def classify_question(state: AgenticRAGState) -> AgenticRAGState:
    """Classify the type of question."""
    prompt = f"""Clasifica esta pregunta en una de las categorías: DEFINITION, BONUS, DEADLINE, APPEAL, OTHER.
    
Pregunta: {state['question']}

Responde solo con la categoría."""
    
    response = llm.invoke(prompt)
    q_type = response.content.strip().upper()
    return {**state, "question_type": q_type}

def reformulate_question(state: AgenticRAGState) -> AgenticRAGState:
    """Reformulate question for better retrieval."""
    prompt = f"""Reformula esta pregunta para mejorar la búsqueda de información.
Mantén la esencia pero hazla más específica.

Pregunta original: {state['question']}

Pregunta reformulada:"""
    
    response = llm.invoke(prompt)
    return {**state, "reformulated_question": response.content.strip()}

def retrieve_context(state: AgenticRAGState) -> AgenticRAGState:
    """Retrieve relevant documents."""
    query = state.get("reformulated_question") or state["question"]
    docs = retriever.invoke(query)
    context = "\n".join([f"- {d.page_content}" for d in docs])
    return {**state, "context": context}

def check_context(state: AgenticRAGState) -> AgenticRAGState:
    """Check if context is sufficient."""
    prompt = f"""¿El contexto contiene información relevante para responder la pregunta?
    
Pregunta: {state['question']}
Contexto: {state['context']}

Responde SI o NO."""
    
    response = llm.invoke(prompt)
    sufficient = "SI" in response.content.upper()
    return {**state, "context_sufficient": sufficient}

def generate_response(state: AgenticRAGState) -> AgenticRAGState:
    """Generate response from context."""
    prompt = f"""Responde la pregunta basándote SOLO en el contexto proporcionado.
Si no hay información suficiente, indícalo claramente.

Contexto:
{state['context']}

Pregunta: {state['question']}

Respuesta:"""
    
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

def check_quality(state: AgenticRAGState) -> AgenticRAGState:
    """Check response quality."""
    prompt = f"""Evalúa la calidad de esta respuesta.
    
Pregunta: {state['question']}
Contexto usado: {state['context']}
Respuesta: {state['response']}

¿La respuesta es BUENA, MEJORABLE o MALA?"""
    
    response = llm.invoke(prompt)
    quality = "BUENA" if "BUENA" in response.content.upper() else "MEJORABLE"
    return {**state, "response_quality": quality, "iteration": state.get("iteration", 0) + 1}

def improve_response(state: AgenticRAGState) -> AgenticRAGState:
    """Improve the response."""
    prompt = f"""Mejora esta respuesta haciéndola más clara y completa.
    
Contexto: {state['context']}
Respuesta original: {state['response']}

Respuesta mejorada:"""
    
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

def finalize(state: AgenticRAGState) -> AgenticRAGState:
    """Finalize the response."""
    return {**state, "final_response": state["response"]}

def no_info_response(state: AgenticRAGState) -> AgenticRAGState:
    """Response when no info found."""
    return {**state, "final_response": "Lo siento, no tengo información suficiente para responder a esa pregunta."}

# Routing functions
def route_context(state: AgenticRAGState) -> Literal["generate", "no_info"]:
    return "generate" if state["context_sufficient"] else "no_info"

def route_quality(state: AgenticRAGState) -> Literal["improve", "finalize"]:
    if state["response_quality"] != "BUENA" and state.get("iteration", 0) < 2:
        return "improve"
    return "finalize"

print("Funciones de nodo definidas ✓")

Funciones de nodo definidas ✓


In [11]:
# Build the graph
workflow = StateGraph(AgenticRAGState)

# Add nodes
workflow.add_node("classify", classify_question)
workflow.add_node("reformulate", reformulate_question)
workflow.add_node("retrieve", retrieve_context)
workflow.add_node("check_context", check_context)
workflow.add_node("generate", generate_response)
workflow.add_node("check_quality", check_quality)
workflow.add_node("improve", improve_response)
workflow.add_node("finalize", finalize)
workflow.add_node("no_info", no_info_response)

# Add edges
workflow.add_edge(START, "classify")
workflow.add_edge("classify", "reformulate")
workflow.add_edge("reformulate", "retrieve")
workflow.add_edge("retrieve", "check_context")
workflow.add_conditional_edges("check_context", route_context, 
                                {"generate": "generate", "no_info": "no_info"})
workflow.add_edge("generate", "check_quality")
workflow.add_conditional_edges("check_quality", route_quality,
                                {"improve": "improve", "finalize": "finalize"})
workflow.add_edge("improve", "check_quality")
workflow.add_edge("finalize", END)
workflow.add_edge("no_info", END)

# Compile with memory
memory = MemorySaver()
agentic_rag = workflow.compile(checkpointer=memory)

print("Agentic RAG compilado ✓")

Agentic RAG compilado ✓


In [12]:
# Test the system
def ask(question: str, session_id: str = "default"):
    config = {"configurable": {"thread_id": session_id}}
    result = agentic_rag.invoke({
        "question": question,
        "question_type": "",
        "reformulated_question": "",
        "context": "",
        "context_sufficient": False,
        "response": "",
        "response_quality": "",
        "final_response": "",
        "iteration": 0
    }, config=config)
    
    print(f"\n📝 Pregunta: {question}")
    print(f"🏷️ Tipo: {result['question_type']}")
    print(f"🔄 Reformulada: {result['reformulated_question']}")
    print(f"✅ Contexto suficiente: {result['context_sufficient']}")
    print(f"⭐ Calidad: {result['response_quality']}")
    print(f"🔢 Iteraciones: {result['iteration']}")
    print(f"\n💬 Respuesta final:\n{result['final_response']}")
    return result

# Test
ask("¿Qué bonificaciones hay para vehículos eléctricos?")


📝 Pregunta: ¿Qué bonificaciones hay para vehículos eléctricos?
🏷️ Tipo: BONUS
🔄 Reformulada: Pregunta reformulada: ¿Cuáles son las bonificaciones y subsidios gubernamentales actuales para la compra y uso de vehículos eléctricos en mi país o región?
✅ Contexto suficiente: True
⭐ Calidad: BUENA
🔢 Iteraciones: 1

💬 Respuesta final:
Hay una bonificación del 75% para vehículos eléctricos en el IVTM.


{'question': '¿Qué bonificaciones hay para vehículos eléctricos?',
 'question_type': 'BONUS',
 'reformulated_question': 'Pregunta reformulada: ¿Cuáles son las bonificaciones y subsidios gubernamentales actuales para la compra y uso de vehículos eléctricos en mi país o región?',
 'context': '- Bonificaciones IVTM: vehículos eléctricos 75%, históricos 100%.\n- El IVTM grava la titularidad de vehículos. Se paga mientras esté matriculado.\n- El IBI se paga anualmente y grava bienes inmuebles urbanos y rústicos.',
 'context_sufficient': True,
 'response': 'Hay una bonificación del 75% para vehículos eléctricos en el IVTM.',
 'response_quality': 'BUENA',
 'final_response': 'Hay una bonificación del 75% para vehículos eléctricos en el IVTM.',
 'iteration': 1}

In [13]:
# Test more questions
questions = [
    "¿Cuándo se paga el IBI?",
    "¿Puedo recurrir una liquidación?",
    "¿Qué descuentos hay para familias numerosas?"
]

for q in questions:
    ask(q)
    print("\n" + "="*60)


📝 Pregunta: ¿Cuándo se paga el IBI?
🏷️ Tipo: DEADLINE
🔄 Reformulada: Pregunta reformulada: ¿Cuál es el plazo y la fecha límite para el pago del Impuesto sobre Bienes Inmuebles (IBI) en España?
✅ Contexto suficiente: True
⭐ Calidad: BUENA
🔢 Iteraciones: 1

💬 Respuesta final:
Según el contexto proporcionado, el IBI se paga anualmente, específicamente en el segundo semestre.


📝 Pregunta: ¿Puedo recurrir una liquidación?
🏷️ Tipo: APPEAL
🔄 Reformulada: Pregunta reformulada: ¿Cuáles son los pasos y requisitos legales para recurrir una liquidación de una empresa o una sentencia de liquidación en un tribunal?
✅ Contexto suficiente: True
⭐ Calidad: BUENA
🔢 Iteraciones: 1

💬 Respuesta final:
Sí, puedes recurrir una liquidación. Tienes 30 días desde la notificación para interponer el recurso.


📝 Pregunta: ¿Qué descuentos hay para familias numerosas?
🏷️ Tipo: OTHER
🔄 Reformulada: Pregunta reformulada: ¿Cuáles son los beneficios y descuentos fiscales o de servicios públicos disponibles para fami

<a name="evaluacion"></a>
## 4. Evaluación con métricas

Evaluamos la calidad del sistema RAG con métricas clave.

In [14]:
# Simple evaluation without RAGAS (to avoid dependency issues)
def evaluate_response(question, response, context, ground_truth=None):
    """Simple evaluation of RAG response quality."""
    
    # Check if response is grounded in context (faithfulness)
    faithfulness_prompt = f"""¿La respuesta está basada en el contexto proporcionado?
    
Contexto: {context}
Respuesta: {response}

Puntúa de 0 a 1 (donde 1 es completamente basada en el contexto)."""
    
    faith_score = llm.invoke(faithfulness_prompt)
    
    # Check relevance
    relevance_prompt = f"""¿La respuesta es relevante para la pregunta?
    
Pregunta: {question}
Respuesta: {response}

Puntúa de 0 a 1."""
    
    rel_score = llm.invoke(relevance_prompt)
    
    return {
        "faithfulness_eval": faith_score.content,
        "relevance_eval": rel_score.content
    }

# Evaluate one response
test_result = ask("¿Qué es el IBI?")
eval_result = evaluate_response(
    test_result["question"],
    test_result["final_response"],
    test_result["context"]
)

print("\n📊 Evaluación:")
print(f"Fidelidad: {eval_result['faithfulness_eval']}")
print(f"Relevancia: {eval_result['relevance_eval']}")


📝 Pregunta: ¿Qué es el IBI?
🏷️ Tipo: DEFINITION
🔄 Reformulada: Pregunta reformulada: ¿Qué es el Impuesto sobre Bienes Inmuebles (IBI) y cómo se calcula en España?
✅ Contexto suficiente: True
⭐ Calidad: BUENA
🔢 Iteraciones: 1

💬 Respuesta final:
El IBI (Impuesto de Bienes Inmuebles) es un impuesto que se paga anualmente y grava bienes inmuebles urbanos y rústicos.

📊 Evaluación:
Fidelidad: La respuesta está completamente basada en el contexto proporcionado. La afirmación sobre el IBI se encuentra directamente en el contexto, por lo que la puntuación sería 1.

La respuesta se basa en la siguiente oración del contexto:
"El IBI se paga anualmente y grava bienes inmuebles urbanos y rústicos."

Por lo tanto, la puntuación es: **1**
Relevancia: La respuesta es completamente relevante para la pregunta, ya que define claramente qué es el IBI y a qué tipo de bienes se aplica. Por lo tanto, le daría una puntuación de 1.


<a name="produccion"></a>
## 5. Mejores prácticas de producción

### Checklist para producción

- [ ] **Logging**: Registrar todas las consultas y respuestas
- [ ] **Métricas**: Monitorear latencia, calidad, uso
- [ ] **Rate limiting**: Controlar costos de API
- [ ] **Fallbacks**: Respuestas por defecto si falla
- [ ] **Caching**: Cachear respuestas frecuentes
- [ ] **Testing**: Tests automatizados de regresión

In [15]:
# Example: Add logging wrapper
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AgenticRAG")

def ask_with_logging(question: str, session_id: str = "default"):
    """Wrapper with logging for production."""
    start_time = datetime.now()
    logger.info(f"Query received: {question}")
    
    try:
        result = ask(question, session_id)
        
        elapsed = (datetime.now() - start_time).total_seconds()
        logger.info(f"Response generated in {elapsed:.2f}s")
        logger.info(f"Quality: {result['response_quality']}")
        
        return result
    except Exception as e:
        logger.error(f"Error processing query: {e}")
        return {"final_response": "Lo siento, ocurrió un error. Intenta de nuevo."}

# Test
ask_with_logging("¿Cuáles son los plazos de pago?")

INFO:AgenticRAG:Query received: ¿Cuáles son los plazos de pago?
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 2.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 2.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:AgenticRAG:Response generated in 5.45s
INFO:Age


📝 Pregunta: ¿Cuáles son los plazos de pago?
🏷️ Tipo: DEADLINE
🔄 Reformulada: Pregunta reformulada: ¿Cuáles son los plazos y condiciones específicas de pago para este servicio o transacción?
✅ Contexto suficiente: True
⭐ Calidad: BUENA
🔢 Iteraciones: 1

💬 Respuesta final:
Los plazos de pago son:
- IBI: segundo semestre
- IVTM: primer trimestre


{'question': '¿Cuáles son los plazos de pago?',
 'question_type': 'DEADLINE',
 'reformulated_question': 'Pregunta reformulada: ¿Cuáles son los plazos y condiciones específicas de pago para este servicio o transacción?',
 'context': '- El IVTM grava la titularidad de vehículos. Se paga mientras esté matriculado.\n- Plazos de pago: IBI segundo semestre, IVTM primer trimestre.\n- Recursos: Puede interponer recurso en 30 días desde la notificación.',
 'context_sufficient': True,
 'response': 'Los plazos de pago son:\n- IBI: segundo semestre\n- IVTM: primer trimestre',
 'response_quality': 'BUENA',
 'final_response': 'Los plazos de pago son:\n- IBI: segundo semestre\n- IVTM: primer trimestre',
 'iteration': 1}

<a name="ejercicio"></a>
## 6. Ejercicio Final

### Mejora el sistema Agentic RAG

Añade las siguientes mejoras:
1. Citación de fuentes en la respuesta
2. Detección de idioma
3. Historial de conversación

In [16]:
# Exercise: Implement improvements
# 1. Modify generate_response to include source citations
# 2. Add a language detection node
# 3. Implement conversation history

In [18]:
# ============================================================
# Exercise: Improved Agentic RAG
# 1. Source citations
# 2. Language detection node
# 3. Conversation history
# ============================================================

from typing import TypedDict, List, Literal


class ImprovedAgenticRAGState(TypedDict):
    question: str
    language: str
    question_type: str
    reformulated_question: str
    context: str
    sources: List[str]
    context_sufficient: bool
    response: str
    response_quality: str
    final_response: str
    iteration: int
    conversation_history: List[str]


# ------------------------------------------------------------
# 1. Language detection node
# ------------------------------------------------------------

def detect_language(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Detect the language of the user question."""
    question = state["question"].lower()

    spanish_markers = [
        "qué", "que", "cuándo", "cuando", "cómo", "como",
        "dónde", "donde", "cuál", "cual", "bonificación",
        "bonificaciones", "impuesto", "paga", "recurso"
    ]

    english_markers = [
        "what", "when", "how", "where", "which",
        "tax", "pay", "appeal", "discount", "bonus"
    ]

    if any(word in question for word in spanish_markers):
        language = "es"
    elif any(word in question for word in english_markers):
        language = "en"
    else:
        language = "es"

    return {
        **state,
        "language": language
    }


# ------------------------------------------------------------
# 2. Classify question
# ------------------------------------------------------------

def classify_question_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Classify the question type."""
    prompt = f"""
Clasifica esta pregunta en una de las siguientes categorías:
DEFINITION, BONUS, DEADLINE, APPEAL, OTHER.

Pregunta:
{state['question']}

Responde solo con la categoría.
"""

    response = llm.invoke(prompt)
    q_type = response.content.strip().upper()

    return {
        **state,
        "question_type": q_type
    }


# ------------------------------------------------------------
# 3. Reformulate using conversation history
# ------------------------------------------------------------

def reformulate_question_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Reformulate the question using conversation history."""
    history = "\n".join(state.get("conversation_history", []))

    prompt = f"""
Reformula la pregunta para mejorar la recuperación de información en un sistema RAG.

Historial de conversación:
{history}

Pregunta actual:
{state['question']}

Instrucciones:
- Mantén la intención original.
- Usa el historial solo si ayuda a resolver referencias como "y eso", "también", "lo anterior", etc.
- Devuelve solo la pregunta reformulada.

Pregunta reformulada:
"""

    response = llm.invoke(prompt)

    return {
        **state,
        "reformulated_question": response.content.strip()
    }


# ------------------------------------------------------------
# 4. Retrieve context with source tracking
# ------------------------------------------------------------

def retrieve_context_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Retrieve relevant documents and keep source citations."""
    query = state.get("reformulated_question") or state["question"]
    docs = retriever.invoke(query)

    context_parts = []
    sources = []

    for i, doc in enumerate(docs, start=1):
        source_id = f"F{i}"
        source_name = doc.metadata.get("source", "unknown")
        source_type = doc.metadata.get("type", "unknown")

        context_parts.append(
            f"[{source_id}] {doc.page_content} "
            f"(source={source_name}, type={source_type})"
        )

        sources.append(
            f"[{source_id}] source={source_name}, type={source_type}"
        )

    context = "\n".join(context_parts)

    return {
        **state,
        "context": context,
        "sources": sources
    }


# ------------------------------------------------------------
# 5. Check context
# ------------------------------------------------------------

def check_context_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Check if the retrieved context is sufficient."""
    prompt = f"""
¿El contexto contiene información relevante para responder la pregunta?

Pregunta:
{state['question']}

Contexto:
{state['context']}

Responde solo SI o NO.
"""

    response = llm.invoke(prompt)
    sufficient = "SI" in response.content.upper()

    return {
        **state,
        "context_sufficient": sufficient
    }


# ------------------------------------------------------------
# 6. Generate response with citations and history
# ------------------------------------------------------------

def generate_response_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Generate response using context, citations and conversation history."""
    history = "\n".join(state.get("conversation_history", []))
    language_instruction = "Responde en español." if state["language"] == "es" else "Answer in English."

    prompt = f"""
Eres un asistente especializado en información tributaria municipal.

{language_instruction}

Usa SOLO el contexto proporcionado.
Incluye citas de fuente usando el formato [F1], [F2], etc.
No inventes información.
Si el contexto no es suficiente, dilo claramente.

Historial de conversación:
{history}

Contexto:
{state['context']}

Pregunta:
{state['question']}

Respuesta con citas:
"""

    response = llm.invoke(prompt)

    return {
        **state,
        "response": response.content
    }


# ------------------------------------------------------------
# 7. Quality check
# ------------------------------------------------------------

def check_quality_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Check response quality."""
    prompt = f"""
Evalúa la calidad de esta respuesta.

Pregunta:
{state['question']}

Contexto:
{state['context']}

Respuesta:
{state['response']}

Criterios:
- Debe responder a la pregunta.
- Debe estar basada en el contexto.
- Debe incluir citas tipo [F1], [F2] si usa información recuperada.
- No debe inventar información.

Responde solo con BUENA, MEJORABLE o MALA.
"""

    response = llm.invoke(prompt)
    quality_text = response.content.upper()

    if "BUENA" in quality_text:
        quality = "BUENA"
    elif "MALA" in quality_text:
        quality = "MALA"
    else:
        quality = "MEJORABLE"

    return {
        **state,
        "response_quality": quality,
        "iteration": state.get("iteration", 0) + 1
    }


# ------------------------------------------------------------
# 8. Improve response if needed
# ------------------------------------------------------------

def improve_response_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Improve response while preserving citations."""
    language_instruction = "Responde en español." if state["language"] == "es" else "Answer in English."

    prompt = f"""
Mejora la respuesta anterior.

{language_instruction}

Debe:
- Ser clara.
- Estar basada solo en el contexto.
- Mantener citas de fuentes tipo [F1], [F2].
- No inventar información.

Contexto:
{state['context']}

Pregunta:
{state['question']}

Respuesta anterior:
{state['response']}

Respuesta mejorada:
"""

    response = llm.invoke(prompt)

    return {
        **state,
        "response": response.content
    }


# ------------------------------------------------------------
# 9. Finalize and update conversation history
# ------------------------------------------------------------

def finalize_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Finalize response and update conversation history."""
    sources_text = "\n".join(state.get("sources", []))

    final_response = state["response"]

    if sources_text:
        if state["language"] == "es":
            final_response = f"{final_response}\n\nFuentes utilizadas:\n{sources_text}"
        else:
            final_response = f"{final_response}\n\nSources used:\n{sources_text}"

    updated_history = state.get("conversation_history", []) + [
        f"Usuario: {state['question']}",
        f"Asistente: {final_response}"
    ]

    return {
        **state,
        "final_response": final_response,
        "conversation_history": updated_history
    }


def no_info_response_improved(state: ImprovedAgenticRAGState) -> ImprovedAgenticRAGState:
    """Response when no useful context is found."""
    if state["language"] == "es":
        final_response = "Lo siento, no tengo información suficiente en el contexto para responder a esa pregunta."
    else:
        final_response = "Sorry, I do not have enough information in the context to answer that question."

    updated_history = state.get("conversation_history", []) + [
        f"Usuario: {state['question']}",
        f"Asistente: {final_response}"
    ]

    return {
        **state,
        "final_response": final_response,
        "conversation_history": updated_history
    }


# ------------------------------------------------------------
# 10. Routing
# ------------------------------------------------------------

def route_context_improved(state: ImprovedAgenticRAGState) -> Literal["generate", "no_info"]:
    return "generate" if state["context_sufficient"] else "no_info"


def route_quality_improved(state: ImprovedAgenticRAGState) -> Literal["improve", "finalize"]:
    if state["response_quality"] != "BUENA" and state.get("iteration", 0) < 2:
        return "improve"
    return "finalize"


# ------------------------------------------------------------
# 11. Build improved graph
# ------------------------------------------------------------

improved_workflow = StateGraph(ImprovedAgenticRAGState)

improved_workflow.add_node("detect_language", detect_language)
improved_workflow.add_node("classify", classify_question_improved)
improved_workflow.add_node("reformulate", reformulate_question_improved)
improved_workflow.add_node("retrieve", retrieve_context_improved)
improved_workflow.add_node("check_context", check_context_improved)
improved_workflow.add_node("generate", generate_response_improved)
improved_workflow.add_node("check_quality", check_quality_improved)
improved_workflow.add_node("improve", improve_response_improved)
improved_workflow.add_node("finalize", finalize_improved)
improved_workflow.add_node("no_info", no_info_response_improved)

improved_workflow.add_edge(START, "detect_language")
improved_workflow.add_edge("detect_language", "classify")
improved_workflow.add_edge("classify", "reformulate")
improved_workflow.add_edge("reformulate", "retrieve")
improved_workflow.add_edge("retrieve", "check_context")

improved_workflow.add_conditional_edges(
    "check_context",
    route_context_improved,
    {
        "generate": "generate",
        "no_info": "no_info"
    }
)

improved_workflow.add_edge("generate", "check_quality")

improved_workflow.add_conditional_edges(
    "check_quality",
    route_quality_improved,
    {
        "improve": "improve",
        "finalize": "finalize"
    }
)

improved_workflow.add_edge("improve", "check_quality")
improved_workflow.add_edge("finalize", END)
improved_workflow.add_edge("no_info", END)

improved_memory = MemorySaver()
improved_agentic_rag = improved_workflow.compile(checkpointer=improved_memory)

print("Improved Agentic RAG compilado correctamente ✓")

Improved Agentic RAG compilado correctamente ✓


In [19]:
# ============================================================
# Test improved Agentic RAG with conversation history
# ============================================================

conversation_history = []

config = {
    "configurable": {
        "thread_id": "exercise_11_demo"
    }
}

questions = [
    "¿Qué es el IBI?",
    "¿Y qué bonificaciones tiene?",
    "When is IVTM paid?"
]

for question in questions:
    result = improved_agentic_rag.invoke(
        {
            "question": question,
            "language": "",
            "question_type": "",
            "reformulated_question": "",
            "context": "",
            "sources": [],
            "context_sufficient": False,
            "response": "",
            "response_quality": "",
            "final_response": "",
            "iteration": 0,
            "conversation_history": conversation_history
        },
        config=config
    )

    conversation_history = result["conversation_history"]

    print("=" * 90)
    print(f"Pregunta: {question}")
    print(f"Idioma detectado: {result['language']}")
    print(f"Tipo de pregunta: {result['question_type']}")
    print(f"Pregunta reformulada: {result['reformulated_question']}")
    print(f"Contexto suficiente: {result['context_sufficient']}")
    print(f"Calidad: {result['response_quality']}")
    print("\nRespuesta final:")
    print(result["final_response"])
    print("\nHistorial acumulado:")
    for h in conversation_history:
        print("-", h[:200])

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Pregunta: ¿Qué es el IBI?
Idioma detectado: es
Tipo de pregunta: DEFINITION
Pregunta reformulada: ¿Qué significa y en qué consiste el Impuesto sobre Bienes Inmuebles (IBI)?
Contexto suficiente: True
Calidad: BUENA

Respuesta final:
El IBI se refiere al Impuesto sobre Bienes Inmuebles, que se paga anualmente y grava bienes inmuebles urbanos y rústicos, según se define en [F1]. Esto significa que es un impuesto que se aplica a la propiedad de bienes inmuebles, ya sean urbanos o rústicos, y se paga cada año.

Fuentes utilizadas:
[F1] source=ibi, type=definition
[F2] source=ibi, type=bonus
[F3] source=ivtm, type=definition

Historial acumulado:
- Usuario: ¿Qué es el IBI?
- Asistente: El IBI se refiere al Impuesto sobre Bienes Inmuebles, que se paga anualmente y grava bienes inmuebles urbanos y rústicos, según se define en [F1]. Esto significa que es un impuesto que se a


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Pregunta: ¿Y qué bonificaciones tiene?
Idioma detectado: es
Tipo de pregunta: BONUS
Pregunta reformulada: ¿Cuáles son las bonificaciones del Impuesto sobre Bienes Inmuebles?
Contexto suficiente: True
Calidad: BUENA

Respuesta final:
Según la información disponible, el IBI ofrece varias bonificaciones. En particular, se mencionan bonificaciones para familias numerosas, que pueden alcanzar hasta un 90% de descuento, y para aquellos que utilizan energías renovables, con un descuento del 50% [F2]. Estas bonificaciones pueden ser beneficiosas para aquellos que cumplen con los requisitos establecidos. Sin embargo, no tengo información adicional sobre otros tipos de bonificaciones o requisitos específicos para acceder a ellas.

Fuentes utilizadas:
[F1] source=ibi, type=definition
[F2] source=ibi, type=bonus
[F3] source=general, type=appeals

Historial acumulado:
- Usuario: ¿Qué es el IBI?
- Asistente: El IBI se refiere al Impuesto sobre Bienes Inmuebles, que se paga anualmente y grava bienes 

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Pregunta: When is IVTM paid?
Idioma detectado: en
Tipo de pregunta: DEADLINE
Pregunta reformulada: ¿Cuándo se paga el Impuesto sobre Vehículos de Tracción Mecánica (IVTM)?
Contexto suficiente: True
Calidad: BUENA

Respuesta final:
Según la información disponible, el IVTM se paga en el primer trimestre del año [F3]. Esto significa que los contribuyentes deben realizar el pago del Impuesto sobre Vehículos de Tracción Mecánica (IVTM) durante los primeros tres meses del año. No tengo información adicional sobre fechas específicas o plazos de pago extendidos.

Sources used:
[F1] source=ivtm, type=definition
[F2] source=ivtm, type=bonus
[F3] source=general, type=deadlines

Historial acumulado:
- Usuario: ¿Qué es el IBI?
- Asistente: El IBI se refiere al Impuesto sobre Bienes Inmuebles, que se paga anualmente y grava bienes inmuebles urbanos y rústicos, según se define en [F1]. Esto significa que es un impuesto que se a
- Usuario: ¿Y qué bonificaciones tiene?
- Asistente: Según la información

## Resumen

En este notebook hemos construido un sistema **Agentic RAG** completo:

1. **Clasificación** de preguntas
2. **Reformulación** para mejor retrieval
3. **Verificación** de contexto suficiente
4. **Generación** de respuestas
5. **Auto-corrección** iterativa
6. **Evaluación** de calidad

Este patrón es la base para asistentes de IA en producción.

En el siguiente y último notebook veremos **Modelos de Gran Contexto y Multimodales**.

---

## Referencias

- [LangGraph Agentic RAG](https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_agentic_rag/)
- [RAGAS Evaluation](https://docs.ragas.io/)

In [20]:
import session_info
session_info.show(html=False)

-----
ipykernel                   6.31.0
langchain_community         0.4.2
langchain_core              1.4.0
langchain_groq              1.1.2
langchain_huggingface       NA
langgraph                   NA
pandas                      2.3.3
session_info                v1.0.1
-----
IPython             9.7.0
jupyter_client      8.6.3
jupyter_core        5.8.1
jupyterlab          4.4.7
notebook            7.4.5
-----
Python 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Windows-10-10.0.19045-SP0
-----
Session information updated at 2026-06-17 18:59
